# Preliminary

Run `aws sso login --profile dev-account` prior to running this

## Setup

In [9]:
import json
import random
import re
import time
from io import StringIO
from pathlib import Path
from typing import Any

import pandas as pd
import requests

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_PMIDS = 10
RANDOM_SEED = 42
REQUIRE_PMID_SPECIFIC_INTERACTION_TYPE = False

DGIDB_URL = "https://www.dgidb.org/data/latest/interactions.tsv"
DGIDB_GRAPHQL_URL = "https://dgidb.org/api/graphql"
DGIDB_GRAPHQL_BATCH_SIZE = 25
DGIDB_GRAPHQL_SLEEP_SECONDS = 0.05
DGIDB_MAX_GRAPHQL_GENES = None
DGIDB_USE_CACHE = True
DGIDB_REFRESH_CACHE = False
DGIDB_CACHE_DIR = OUT_DIR / "dgidb_cache"
DGIDB_SUMMARY_CACHE_PATH = DGIDB_CACHE_DIR / "interactions.tsv"
DGIDB_SOURCE_CACHE_PATH = DGIDB_CACHE_DIR / "publication_interactions.csv"
DGIDB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

MECHANISTIC_TYPES = {"inhibitor", "activator"}
TYPE_ALIASES = {
    "inhibitor": "inhibitor",
    "inhibitors": "inhibitor",
    "inhibit": "inhibitor",
    "inhibits": "inhibitor",
    "inhibited": "inhibitor",
    "inhibiting": "inhibitor",
    "inhibition": "inhibitor",
    "inhibitory": "inhibitor",
    "activator": "activator",
    "activators": "activator",
    "activate": "activator",
    "activates": "activator",
    "activated": "activator",
    "activating": "activator",
    "activation": "activator",
}


In [10]:
def clean_value(value: Any) -> str:
    text = "" if pd.isna(value) else str(value).strip()
    return "" if text.upper() in {"", "NULL", "NA", "N/A", "NONE", "NAN"} else text


def norm_text(value: Any) -> str:
    text = "" if pd.isna(value) else str(value).lower().strip()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text)).strip()


def norm_gene(value: Any) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "", "" if pd.isna(value) else str(value)).upper()


def norm_col(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")


def find_col(df: pd.DataFrame, names: list[str]) -> str:
    lookup = {norm_col(c): c for c in df.columns}
    for name in names:
        key = norm_col(name)
        if key in lookup:
            return lookup[key]
    raise ValueError(f"Missing column. Tried {names}. Found {list(df.columns)}")


def trueish(value: Any) -> bool:
    return norm_text(value) in {"1", "true", "yes", "y", "include", "included"}


def canonical_interaction_type(value: Any) -> str:
    return TYPE_ALIASES.get(norm_text(value), "")


def read_table(source: str | Path, sep: str = "	") -> pd.DataFrame:
    source = str(source)
    if source.startswith(("http://", "https://")):
        response = requests.get(source, headers={"User-Agent": "dgilit-pmid-eval"}, timeout=120)
        response.raise_for_status()
        if "<html" in response.text[:1000].lower():
            raise ValueError(f"Expected a delimited data file, received HTML from {source}")
        return pd.read_csv(StringIO(response.text), sep=sep, dtype=str, keep_default_na=False)
    return pd.read_csv(source, sep=sep, dtype=str, keep_default_na=False)


def read_cached_table(
    source: str | Path,
    cache_path: Path,
    sep: str = "	",
    refresh: bool = DGIDB_REFRESH_CACHE,
    use_cache: bool = DGIDB_USE_CACHE,
) -> pd.DataFrame:
    if use_cache and cache_path.exists() and not refresh:
        return pd.read_csv(cache_path, sep=sep, dtype=str, keep_default_na=False)
    df = read_table(source, sep=sep)
    if use_cache:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(cache_path, sep=sep, index=False)
    return df


def out_path(name: str, suffix: str = "csv", n_pmids: int = N_PMIDS) -> Path:
    return OUT_DIR / f"{name}_{n_pmids}.{suffix}"


## Load DGIdb

In [11]:
DGIDB_GRAPHQL_QUERY = """
query getInteractionsByGene($names: [String!]) {
  genes(names: $names) {
    nodes {
      interactions {
        drug { name }
        gene { name }
        interactionTypes { type directionality }
        publications { pmid }
        interactionClaims { publications { pmid } }
      }
    }
  }
}
"""


def standardize_summary(raw: pd.DataFrame) -> pd.DataFrame:
    drug_col = find_col(raw, ["drug_name", "drug", "drug_claim_primary_name", "drug_claim_name"])
    gene_col = find_col(raw, ["gene_name", "gene", "gene_claim_name"])
    summary = pd.DataFrame({
        "drug_name": raw[drug_col].map(clean_value),
        "gene_name": raw[gene_col].map(clean_value),
    })
    summary = summary[(summary["drug_name"] != "") & (summary["gene_name"] != "")]
    return summary.drop_duplicates().reset_index(drop=True)


def graphql_post(session: requests.Session, query: str, variables: dict) -> dict:
    response = session.post(
        DGIDB_GRAPHQL_URL,
        json={"query": query, "variables": variables},
        headers={"User-Agent": "dgilit-pmid-eval", "dgidb-client-name": "dgilit-pmid-eval"},
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("errors"):
        raise RuntimeError(json.dumps(payload["errors"], indent=2)[:4000])
    return payload


def publication_pmids(publications: list[dict] | None) -> set[str]:
    return {
        clean_value((pub or {}).get("pmid"))
        for pub in (publications or [])
        if clean_value((pub or {}).get("pmid"))
    }


def interaction_types_from(items: list[dict] | None) -> list[str]:
    return sorted({
        canonical_interaction_type((item or {}).get("type"))
        for item in (items or [])
        if canonical_interaction_type((item or {}).get("type"))
    })


def rows_from_interaction(interaction: dict) -> list[dict]:
    gene_name = clean_value((interaction.get("gene") or {}).get("name"))
    drug_name = clean_value((interaction.get("drug") or {}).get("name"))
    if not gene_name or not drug_name:
        return []

    interaction_types = interaction_types_from(interaction.get("interactionTypes"))
    interaction_pmids = publication_pmids(interaction.get("publications"))
    claim_pmids = {
        pmid
        for claim in (interaction.get("interactionClaims") or [])
        for pmid in publication_pmids((claim or {}).get("publications"))
    }
    pmids = sorted(interaction_pmids | claim_pmids)
    if not pmids:
        return []

    pmid_specific = len(pmids) == 1 and bool(interaction_types)
    types_to_emit = interaction_types if interaction_types else [""]

    rows = []
    for pmid in pmids:
        for interaction_type in types_to_emit:
            rows.append({
                "pmid": pmid,
                "drug_name": drug_name,
                "gene_name": gene_name,
                "interaction_type": interaction_type,
                "interaction_type_pmid_specific": pmid_specific,
            })
    return rows


def fetch_publication_interactions(genes: list[str]) -> pd.DataFrame:
    genes = [g for g in dict.fromkeys(clean_value(g) for g in genes) if g]
    if DGIDB_MAX_GRAPHQL_GENES is not None:
        genes = genes[:int(DGIDB_MAX_GRAPHQL_GENES)]

    rows = []
    session = requests.Session()
    for start in tqdm(range(0, len(genes), DGIDB_GRAPHQL_BATCH_SIZE), desc="DGIdb GraphQL gene batches"):
        batch = genes[start:start + DGIDB_GRAPHQL_BATCH_SIZE]
        payload = graphql_post(session, DGIDB_GRAPHQL_QUERY, {"names": batch})
        for node in payload.get("data", {}).get("genes", {}).get("nodes") or []:
            for interaction in node.get("interactions") or []:
                rows.extend(rows_from_interaction(interaction))
        if DGIDB_GRAPHQL_SLEEP_SECONDS:
            time.sleep(DGIDB_GRAPHQL_SLEEP_SECONDS)

    columns = ["pmid", "drug_name", "gene_name", "interaction_type", "interaction_type_pmid_specific"]
    return pd.DataFrame(rows, columns=columns).drop_duplicates().reset_index(drop=True)


def load_cached_dgidb_source(
    refresh: bool = DGIDB_REFRESH_CACHE,
    use_cache: bool = DGIDB_USE_CACHE,
) -> pd.DataFrame:
    if use_cache and DGIDB_SOURCE_CACHE_PATH.exists() and not refresh:
        return pd.read_csv(DGIDB_SOURCE_CACHE_PATH, dtype=str, keep_default_na=False)

    raw = read_cached_table(DGIDB_URL, DGIDB_SUMMARY_CACHE_PATH, sep="	", refresh=refresh, use_cache=use_cache)
    summary = standardize_summary(raw)
    genes = sorted(summary["gene_name"].dropna().unique())
    if not genes:
        raise ValueError("No genes found in DGIdb interaction summary TSV.")

    source = fetch_publication_interactions(genes)
    if use_cache:
        DGIDB_SOURCE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        source.to_csv(DGIDB_SOURCE_CACHE_PATH, index=False)
    return source


source_df = load_cached_dgidb_source()
eligible_df = source_df[
    (
        source_df["interaction_type_pmid_specific"].map(trueish)
        | (not REQUIRE_PMID_SPECIFIC_INTERACTION_TYPE)
    )
    & source_df["interaction_type"].isin(MECHANISTIC_TYPES)
].copy().reset_index(drop=True)

if eligible_df.empty:
    raise ValueError("No eligible inhibitor/activator DGIdb interactions available.")

eligible_df.head()


,pmid,drug_name,gene_name,interaction_type,interaction_type_pmid_specific
0,21657271,TARIQUIDAR,ABCB1,inhibitor,True
1,15456083,VALSPODAR,ABCB1,inhibitor,False
2,9446255,VALSPODAR,ABCB1,inhibitor,False
3,16805961,CYCLOSPORINE,ABCG2,inhibitor,False
4,17031644,CYCLOSPORINE,ABCG2,inhibitor,False


## Filter eligible PMIDs to those with PubMed abstracts

PMIDs without a usable PubMed abstract are removed from `eligible_df` before sampling, so downstream answer sheets and prompts are built only from abstract-backed PMIDs.


In [ ]:
PUBMED_EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
PUBMED_ABSTRACT_CACHE_PATH = OUT_DIR / "pubmed_abstracts.csv"
PUBMED_BATCH_SIZE = 100
PUBMED_SLEEP_SECONDS = 0.34

NCBI_TOOL = "dgilit-pmid-eval"
NCBI_EMAIL = ""


def collapse_ws(text: Any) -> str:
    return re.sub(r"\s+", " ", "" if text is None else str(text)).strip()


def element_text(element: Any) -> str:
    if element is None:
        return ""
    return collapse_ws("".join(element.itertext()))


def parse_pubmed_abstracts(xml_text: str) -> pd.DataFrame:
    import xml.etree.ElementTree as ET

    root = ET.fromstring(xml_text)
    rows = []

    for article in root.findall(".//PubmedArticle"):
        pmid = clean_value(element_text(article.find(".//MedlineCitation/PMID")))
        title = element_text(article.find(".//ArticleTitle"))

        abstract_parts = []
        for abstract_text in article.findall(".//Abstract/AbstractText"):
            text = element_text(abstract_text)
            if not text:
                continue
            label = clean_value(abstract_text.attrib.get("Label") or abstract_text.attrib.get("NlmCategory") or "")
            abstract_parts.append(f"{label}: {text}" if label else text)

        rows.append({
            "pmid": pmid,
            "article_title": title,
            "abstract": "\n".join(abstract_parts).strip(),
        })

    return pd.DataFrame(rows, columns=["pmid", "article_title", "abstract"]).drop_duplicates("pmid")


def fetch_pubmed_abstract_batch(pmids: list[str]) -> pd.DataFrame:
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "tool": NCBI_TOOL,
    }
    if NCBI_EMAIL:
        params["email"] = NCBI_EMAIL

    response = requests.get(
        PUBMED_EFETCH_URL,
        params=params,
        headers={"User-Agent": NCBI_TOOL},
        timeout=120,
    )
    response.raise_for_status()
    return parse_pubmed_abstracts(response.text)


def fetch_pubmed_abstracts(
    pmids: list[str],
    cache_path: Path = PUBMED_ABSTRACT_CACHE_PATH,
    refresh: bool = False,
    use_cache: bool = True,
) -> pd.DataFrame:
    pmids = [str(pmid).strip() for pmid in dict.fromkeys(pmids) if str(pmid).strip()]
    columns = ["pmid", "article_title", "abstract"]

    if use_cache and cache_path.exists() and not refresh:
        cached = pd.read_csv(cache_path, dtype=str, keep_default_na=False)
        for col in columns:
            if col not in cached.columns:
                cached[col] = ""
        cached = cached[columns].drop_duplicates("pmid")
    else:
        cached = pd.DataFrame(columns=columns)

    cached_pmids = set(cached["pmid"].astype(str))
    missing_pmids = [pmid for pmid in pmids if pmid not in cached_pmids]

    fetched_frames = []
    for start in tqdm(range(0, len(missing_pmids), PUBMED_BATCH_SIZE), desc="PubMed abstract batches"):
        batch = missing_pmids[start:start + PUBMED_BATCH_SIZE]
        if not batch:
            continue

        fetched = fetch_pubmed_abstract_batch(batch)
        fetched_pmids = set(fetched["pmid"].astype(str)) if not fetched.empty else set()
        missing_from_response = [
            {"pmid": pmid, "article_title": "", "abstract": ""}
            for pmid in batch
            if pmid not in fetched_pmids
        ]

        if missing_from_response:
            fetched = pd.concat([fetched, pd.DataFrame(missing_from_response)], ignore_index=True)

        fetched_frames.append(fetched[columns])
        if PUBMED_SLEEP_SECONDS:
            time.sleep(PUBMED_SLEEP_SECONDS)

    if fetched_frames:
        cached = pd.concat([cached, *fetched_frames], ignore_index=True)
        cached = cached[columns].fillna("").drop_duplicates("pmid", keep="last")
        if use_cache:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            cached.to_csv(cache_path, index=False)

    return cached[cached["pmid"].astype(str).isin(pmids)].copy().reset_index(drop=True)


pre_abstract_filter_pmid_count = eligible_df["pmid"].astype(str).nunique()
eligible_pmids = sorted(eligible_df["pmid"].astype(str).unique())
eligible_abstracts_df = fetch_pubmed_abstracts(eligible_pmids)

has_abstract = eligible_abstracts_df["abstract"].map(lambda value: collapse_ws(value) != "")
eligible_abstracts_df = eligible_abstracts_df.loc[has_abstract, ["pmid", "article_title", "abstract"]].copy()
eligible_abstract_pmids = set(eligible_abstracts_df["pmid"].astype(str))

eligible_df = eligible_df[
    eligible_df["pmid"].astype(str).isin(eligible_abstract_pmids)
].copy().reset_index(drop=True)

post_abstract_filter_pmid_count = eligible_df["pmid"].astype(str).nunique()
removed_no_abstract_count = pre_abstract_filter_pmid_count - post_abstract_filter_pmid_count

if eligible_df.empty:
    raise ValueError("No eligible inhibitor/activator DGIdb interactions with PubMed abstracts are available.")
if post_abstract_filter_pmid_count < N_PMIDS:
    raise ValueError(
        f"Only {post_abstract_filter_pmid_count} eligible PMID(s) have PubMed abstracts; cannot sample {N_PMIDS}."
    )

pd.DataFrame([
    {
        "pmids_before_abstract_filter": pre_abstract_filter_pmid_count,
        "pmids_removed_without_abstract": removed_no_abstract_count,
        "pmids_after_abstract_filter": post_abstract_filter_pmid_count,
        "eligible_interaction_rows_after_filter": len(eligible_df),
    }
])


PubMed abstract batches: 0it [00:00, ?it/s]


,pmids_before_abstract_filter,pmids_removed_without_abstract,pmids_after_abstract_filter,eligible_interaction_rows_after_filter
0,2072,110,1962,4168


## Build answer sheet

In [13]:
def simple_random_sample_pmids(df: pd.DataFrame, n: int, seed: int) -> list[str]:
    pmids = sorted(df["pmid"].astype(str).unique())
    if len(pmids) < n:
        raise ValueError(f"Only {len(pmids)} eligible PMIDs are available; cannot sample {n}.")
    return sorted(random.Random(seed).sample(pmids, n))


selected_pmids = simple_random_sample_pmids(eligible_df, N_PMIDS, RANDOM_SEED)

answer_sheet_full = eligible_df[eligible_df["pmid"].astype(str).isin(selected_pmids)].copy()
answer_sheet_full = answer_sheet_full[
    (
        answer_sheet_full["interaction_type_pmid_specific"].map(trueish)
        | (not REQUIRE_PMID_SPECIFIC_INTERACTION_TYPE)
    )
    & answer_sheet_full["interaction_type"].isin(MECHANISTIC_TYPES)
].copy()
answer_sheet_full = answer_sheet_full.drop_duplicates().sort_values(
    ["pmid", "interaction_type", "gene_name", "drug_name"]
).reset_index(drop=True)

if answer_sheet_full.empty:
    raise ValueError("Answer sheet is empty after typed filtering.")
if answer_sheet_full["pmid"].nunique() != N_PMIDS:
    raise ValueError("Answer sheet does not contain exactly the requested number of PMIDs.")
if REQUIRE_PMID_SPECIFIC_INTERACTION_TYPE and not answer_sheet_full["interaction_type_pmid_specific"].map(trueish).all():
    raise ValueError("Answer sheet contains a non-PMID-specific interaction type.")
if not answer_sheet_full["interaction_type"].isin(MECHANISTIC_TYPES).all():
    raise ValueError("Answer sheet contains a disallowed interaction type.")

answer_sheet = answer_sheet_full[["pmid", "drug_name", "gene_name", "interaction_type"]].copy()
answer_sheet_path = out_path("answer_sheet")
answer_sheet.to_csv(answer_sheet_path, index=False)

answer_sheet.head()


,pmid,drug_name,gene_name,interaction_type
0,12799093,ENALAPRIL MALEATE,ACE,inhibitor
1,17638512,THIOTEPA,CYP2B6,inhibitor
2,17638512,TICLOPIDINE,CYP2B6,inhibitor
3,18089823,CETUXIMAB,EGFR,inhibitor
4,18923523,CRIZOTINIB,ALK,inhibitor


## Create prompts with PubMed abstracts


In [14]:
def build_prompt(pmid: str, abstract: str, article_title: str = "") -> str:
    article_title = collapse_ws(article_title)
    abstract = str(abstract or "").strip()

    return f"""
Given the PMID and PubMed abstract below, identify drug-gene interactions supported by the abstract where the drug/gene relationship is specifically inhibiting or activating.

Return only valid JSON. Do not include markdown, explanation, or prose.

JSON shape:
[
  {{
    "pmid": "{pmid}",
    "drug_name": "drug name",
    "gene_name": "gene symbol/name",
    "interaction_type": "inhibitor or activator"
  }}
]

Rules:
- Base your answer only on the PMID and abstract provided in this prompt.
- Use interaction_type "inhibitor" only for inhibiting/inhibition/inhibits language.
- Use interaction_type "activator" only for activating/activation/activates language.
- Return [] if the abstract does not support any inhibitor or activator drug-gene interaction.

PMID: {pmid}
Article title: {article_title}

Abstract:
{abstract}
""".strip()


task_pmids = pd.DataFrame({"pmid": sorted(answer_sheet["pmid"].astype(str).unique())})
task_pmids = task_pmids.merge(eligible_abstracts_df, on="pmid", how="left").fillna("")

missing_prompt_pmids = task_pmids.loc[
    task_pmids["abstract"].map(lambda value: collapse_ws(value) == ""), "pmid"
].tolist()
if missing_prompt_pmids:
    raise ValueError(
        "Internal error: sampled PMID(s) are missing abstracts after eligible_df filtering: "
        f"{missing_prompt_pmids}"
    )

prompts = [
    {
        "pmid": row.pmid,
        "article_title": row.article_title,
        "abstract": row.abstract,
        "prompt": build_prompt(row.pmid, row.abstract, row.article_title),
    }
    for row in task_pmids.itertuples(index=False)
]

pmid_tasks_path = out_path("pmid_tasks_with_abstracts")
prompts_path = out_path("prompts_with_abstracts", "json")

task_pmids.to_csv(pmid_tasks_path, index=False)
prompts_path.write_text(json.dumps(prompts, indent=2, ensure_ascii=False) + "\n")

prompts[:2]


[{'pmid': '12799093',
  'article_title': 'Effect of quinapril, quinapril-hydrochlorothiazide, and enalapril on the bone mass of hypertensive subjects: relationship with angiotensin converting enzyme polymorphisms.',
  'abstract': 'BACKGROUND: Many alterations in extracellular metabolism of calcium have been associated to hypertension, but the number of studies relating this disease with osteoporosis is extremely low. This study clarifies the therapeutic effect of three treatments-quinapril, quinapril + hydrochlorothiazide (HCTZ), enalapril-on bone remodeling markers, bone mineral density (BMD) in hypertensive patients, and relationship with angiotensin converting enzyme (ACE) polymorphism.\nMETHODS: This open, prospective study included 134 patients with low-to-moderate hypertension and stable BMD according to Joint National Committee criteria and 96 patients completed the study. After a washout period, patients were randomized to one of the three treatments, which they received for 1 

## Run Bedrock predictions

In [15]:
BEDROCK_REGION = "us-east-2"
BEDROCK_PROFILE_NAME = "dev-account"
BEDROCK_MODEL_KEY = "claude-sonnet-4-6"
BEDROCK_MODELS = {
    "claude-sonnet-4-6": "us.anthropic.claude-sonnet-4-6",
    "claude-sonnet-4-5": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "claude-3-5-sonnet": "us.anthropic.claude-3-5-sonnet-20240620-v1:0",
}
BEDROCK_MAX_TOKENS = 2048
BEDROCK_TEMPERATURE = 0.0
BEDROCK_SLEEP_SECONDS = 0.25
BEDROCK_RETRIES = 3
BEDROCK_OVERWRITE = False
BEDROCK_LIMIT = None
BEDROCK_STOP_ON_ERROR = False
RUN_BEDROCK = True

BEDROCK_SYSTEM_PROMPT = "You are an expert biomedical curator. Return only valid JSON. Do not include markdown, explanations, caveats, or chain-of-thought."

predictions_path = out_path(f"predictions_{BEDROCK_MODEL_KEY}", "json")


def bedrock_client(region_name: str = BEDROCK_REGION):
    import boto3

    session_kwargs = {"profile_name": BEDROCK_PROFILE_NAME} if BEDROCK_PROFILE_NAME else {}
    return boto3.Session(**session_kwargs).client("bedrock-runtime", region_name=region_name)


def bedrock_request_body(prompt: str, max_tokens: int = BEDROCK_MAX_TOKENS, temperature: float = BEDROCK_TEMPERATURE) -> dict:
    return {
        "anthropic_version": "bedrock-2023-05-31",
        "system": BEDROCK_SYSTEM_PROMPT,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
    }


def decode_bedrock_body(body: Any) -> dict:
    raw = body.read() if hasattr(body, "read") else body
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    if isinstance(raw, str):
        return json.loads(raw)
    return raw or {}


def bedrock_response_text(response: dict) -> str:
    if "output" in response:
        blocks = response.get("output", {}).get("message", {}).get("content", [])
        return "\n".join(block.get("text", "") for block in blocks if isinstance(block, dict) and block.get("text")).strip()

    payload = decode_bedrock_body(response.get("body", response))
    blocks = payload.get("content", [])
    text = "\n".join(
        block.get("text", "")
        for block in blocks
        if isinstance(block, dict) and block.get("text")
    ).strip()
    return text or str(payload.get("completion", "")).strip()


def invoke_bedrock_prompt(
    prompt: str,
    client: Any | None = None,
    model_id: str | None = None,
    region_name: str = BEDROCK_REGION,
    max_tokens: int = BEDROCK_MAX_TOKENS,
    temperature: float = BEDROCK_TEMPERATURE,
    retries: int = BEDROCK_RETRIES,
) -> str:
    client = client or bedrock_client(region_name)
    model_id = model_id or BEDROCK_MODELS[BEDROCK_MODEL_KEY]
    body = bedrock_request_body(prompt, max_tokens=max_tokens, temperature=temperature)
    for attempt in range(1, retries + 1):
        try:
            response = client.invoke_model(
                modelId=model_id,
                body=json.dumps(body),
                contentType="application/json",
                accept="application/json",
            )
            return bedrock_response_text(response)
        except Exception:
            if attempt == retries:
                raise
            time.sleep(BEDROCK_SLEEP_SECONDS * attempt)
    return ""


def json_payloads_from_text(text: str) -> list[Any]:
    text = str(text or "").strip()
    if not text:
        return []

    candidates = []
    candidates.extend(match.group(1) for match in re.finditer(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.I | re.S))
    candidates.append(re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I).strip())

    decoder = json.JSONDecoder()
    payloads = []
    seen = set()
    for candidate in candidates:
        starts = [0] + [match.start() for match in re.finditer(r"[\[{]", candidate)]
        for start in dict.fromkeys(starts):
            fragment = candidate[start:].strip()
            if not fragment:
                continue
            try:
                payload, _ = decoder.raw_decode(fragment)
            except json.JSONDecodeError:
                continue
            key = json.dumps(payload, sort_keys=True, default=str)
            if key not in seen:
                seen.add(key)
                payloads.append(payload)
    return payloads


def extract_json_objects(text: str) -> list[dict]:
    empty_result = []
    for payload in json_payloads_from_text(text):
        if isinstance(payload, dict) and isinstance(payload.get("interactions"), list):
            objects = payload["interactions"]
        elif isinstance(payload, list):
            objects = payload
        elif isinstance(payload, dict):
            objects = [payload]
        else:
            objects = []

        rows = [obj for obj in objects if isinstance(obj, dict)]
        if rows:
            return rows
        if isinstance(payload, list) and not payload:
            empty_result = []
    return empty_result


def clean_prediction_payload(pmid: str, payload: Any) -> list[dict]:
    if isinstance(payload, dict):
        objects = payload.get("interactions", [])
        if not objects and payload.get("raw_response"):
            objects = extract_json_objects(payload["raw_response"])
    elif isinstance(payload, list):
        objects = payload
    else:
        objects = []

    allowed_pmids = set(task_pmids["pmid"].astype(str))
    rows = []
    for obj in objects:
        if not isinstance(obj, dict):
            continue
        row = {
            "pmid": str(obj.get("pmid") or pmid).strip(),
            "drug_name": clean_value(obj.get("drug_name") or obj.get("drug") or ""),
            "gene_name": clean_value(obj.get("gene_name") or obj.get("gene") or ""),
            "interaction_type": canonical_interaction_type(obj.get("interaction_type") or obj.get("type") or ""),
        }
        if row["pmid"] in allowed_pmids and row["drug_name"] and row["gene_name"] and row["interaction_type"] in MECHANISTIC_TYPES:
            rows.append(row)
    return rows


def load_predictions(path: Path) -> dict:
    if path.exists():
        return json.loads(path.read_text())
    return {}


def save_predictions(predictions: dict, path: Path = predictions_path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(predictions, indent=2, ensure_ascii=False) + "\n")
    return path


def prediction_record(pmid: str, model_key: str, model_id: str, raw_response: str, error: str = "") -> dict:
    return {
        "model_key": model_key,
        "model_id": model_id,
        "raw_response": raw_response,
        "interactions": clean_prediction_payload(pmid, {"raw_response": raw_response}),
        "error": error,
    }


def repair_prediction_record(pmid: str, payload: dict) -> dict:
    raw_response = str(payload.get("raw_response", "")) if isinstance(payload, dict) else ""
    interactions = clean_prediction_payload(pmid, payload)
    return {
        **payload,
        "raw_response": raw_response,
        "interactions": interactions,
        "error": str(payload.get("error", "")),
    }


def run_bedrock_predictions(
    prompts: list[dict],
    predictions_path: Path = predictions_path,
    model_key: str = BEDROCK_MODEL_KEY,
    model_id: str | None = None,
    region_name: str = BEDROCK_REGION,
    overwrite: bool = BEDROCK_OVERWRITE,
    limit: int | None = BEDROCK_LIMIT,
    stop_on_error: bool = BEDROCK_STOP_ON_ERROR,
) -> dict:
    model_id = model_id or BEDROCK_MODELS[model_key]
    client = bedrock_client(region_name) if RUN_BEDROCK else None
    predictions = load_predictions(predictions_path)
    items = prompts[:limit] if limit else prompts

    for item in tqdm(items, desc=f"Bedrock {model_key}"):
        pmid = str(item["pmid"])
        existing = predictions.get(pmid, {})
        if isinstance(existing, dict) and existing.get("raw_response") and not overwrite:
            predictions[pmid] = repair_prediction_record(pmid, existing)
            save_predictions(predictions, predictions_path)
            continue
        if not RUN_BEDROCK:
            continue
        try:
            raw_response = invoke_bedrock_prompt(item["prompt"], client=client, model_id=model_id)
            error = ""
        except Exception as exc:
            if stop_on_error:
                raise
            raw_response = ""
            error = str(exc)
        predictions[pmid] = prediction_record(pmid, model_key, model_id, raw_response, error)
        save_predictions(predictions, predictions_path)
        if BEDROCK_SLEEP_SECONDS:
            time.sleep(BEDROCK_SLEEP_SECONDS)
    return predictions


predictions = run_bedrock_predictions(prompts)
save_predictions(predictions, predictions_path)

prediction_status = pd.DataFrame([
    {
        "pmid": pmid,
        "raw_response_chars": len(str(payload.get("raw_response", ""))),
        "parsed_interactions": len(payload.get("interactions", [])),
        "error": payload.get("error", ""),
    }
    for pmid, payload in sorted(predictions.items())
])

prediction_status

Bedrock claude-sonnet-4-6: 100%|██████████| 10/10 [00:28<00:00,  2.83s/it]


,pmid,raw_response_chars,parsed_interactions,error
0,12799093,248,2,
1,17638512,2,0,
2,18089823,501,4,
3,18923523,2,0,
4,21120162,2,0,
5,21594703,2,0,
6,22238366,629,5,
7,29573941,379,3,
8,33268594,124,1,
9,33495309,380,3,


## Score predictions

In [16]:
def predictions_to_frame(predictions: dict) -> pd.DataFrame:
    rows = []
    for pmid, payload in predictions.items():
        rows.extend(clean_prediction_payload(str(pmid), payload))
    return pd.DataFrame(rows, columns=["pmid", "drug_name", "gene_name", "interaction_type"]).fillna("").drop_duplicates()


def standardize_interactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().fillna("")
    for col in ["pmid", "drug_name", "gene_name", "interaction_type"]:
        if col not in df.columns:
            df[col] = ""
    df["pmid_key"] = df["pmid"].astype(str).str.strip()
    df["drug_key"] = df["drug_name"].map(norm_text)
    df["gene_key"] = df["gene_name"].map(norm_gene)
    df["type_key"] = df["interaction_type"].map(canonical_interaction_type)
    return df[(df[["pmid_key", "drug_key", "gene_key"]] != "").all(axis=1) & df["type_key"].isin(MECHANISTIC_TYPES)]


def tuple_set(df: pd.DataFrame, cols: list[str]) -> set[tuple]:
    return set() if df.empty else set(map(tuple, df[cols].drop_duplicates().to_numpy()))


def compute_metrics(expected_df: pd.DataFrame, predicted_df: pd.DataFrame, cols: list[str], level: str) -> dict:
    expected = tuple_set(expected_df, cols)
    predicted = tuple_set(predicted_df, cols)
    tp = expected & predicted
    fp = predicted - expected
    fn = expected - predicted
    precision = len(tp) / len(predicted) if predicted else 0.0
    recall = len(tp) / len(expected) if expected else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "level": level,
        "expected": len(expected),
        "predicted": len(predicted),
        "true_positives": len(tp),
        "false_positives": len(fp),
        "false_negatives": len(fn),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def score_predictions(answer_sheet: pd.DataFrame, predictions_path: Path = predictions_path) -> pd.DataFrame:
    predictions_path = Path(predictions_path)
    if not predictions_path.exists():
        raise FileNotFoundError(f"Prediction file does not exist: {predictions_path}")

    predictions = load_predictions(predictions_path)
    predictions = {pmid: repair_prediction_record(str(pmid), payload) for pmid, payload in predictions.items()}
    save_predictions(predictions, predictions_path)
    pred_df = predictions_to_frame(predictions)

    answer_std = standardize_interactions(answer_sheet)
    pred_std = standardize_interactions(pred_df)

    answer_pair = answer_std.drop_duplicates(subset=["pmid_key", "drug_key", "gene_key"])
    answer_typed = answer_std.drop_duplicates(subset=["pmid_key", "drug_key", "gene_key", "type_key"])
    pred_pair = pred_std.drop_duplicates(subset=["pmid_key", "drug_key", "gene_key"])
    pred_typed = pred_std.drop_duplicates(subset=["pmid_key", "drug_key", "gene_key", "type_key"])

    metrics = pd.DataFrame([
        compute_metrics(answer_pair, pred_pair, ["pmid_key", "drug_key", "gene_key"], "pair"),
        compute_metrics(answer_typed, pred_typed, ["pmid_key", "drug_key", "gene_key", "type_key"], "typed"),
    ])
    metrics.to_csv(out_path("metrics"), index=False)
    return metrics


metrics_df = score_predictions(answer_sheet)
metrics_df

,level,expected,predicted,true_positives,false_positives,false_negatives,precision,recall,f1
0,pair,27,18,4,14,23,0.222222,0.148148,0.177778
1,typed,28,18,4,14,24,0.222222,0.142857,0.173913
